# Cache Encoder Features (COCO → ResNet50 2048-d vectors)

**One-time pre-processing step.** Runs ResNet50 over every COCO image once,
saves pooled 2048-d fp16 vectors to disk.

**Before running:** Add these Datasets:
- `awsaf49/coco-2017-dataset` (or your COCO images)
- `shtvkumar/karpathy-splits`

**After running:** Upload `/kaggle/working/features/` as a new Kaggle Dataset.
Mount it in your training notebook.

In [ ]:
# P100 (sm_60) needs torch compiled with CUDA 11.8, not 12.x
!pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118 -q
import torch; print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
import os, sys

REPO_URL = "https://github.com/MohamedShakshak/Image-Captioning.git"

if not os.path.isdir("Image-Captioning"):
    !git clone --depth 1 {REPO_URL}
%cd Image-Captioning

!pip install -e . -q

_src = os.path.join(os.getcwd(), "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
# SET THESE to match your Kaggle mounts
COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
KARPATHY_SPLIT = "/kaggle/input/datasets/shtvkumar/karpathy-splits/dataset_coco.json"
OUTPUT_DIR = "/kaggle/working/features"

for p in [COCO_ROOT, KARPATHY_SPLIT]:
    assert os.path.exists(p), f"Missing: {p}"
    print(f"OK: {p}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Path normalization

The Karpathy split references COCO 2014 filenames (`val2014/COCO_val2014_*.jpg`)
but the mounted images are COCO 2017 (`val2017/000000*.jpg`). These are the **same
images** with different naming. The cell below normalizes the paths by stripping
the 2014 prefix and mapping to 2017 directories.

In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
from data.transforms import inference_transforms
from models.encoder import Encoder

# Load Karpathy split
with open(KARPATHY_SPLIT) as f:
    split = json.load(f)

# Normalize: strip "COCO_train2014_" / "COCO_val2014_" prefix,
# map train2014->train2017, val2014->val2017
COCO_YEAR_MAP = {"train2014": "train2017", "val2014": "val2017"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

encoder = Encoder().to(device).eval()
transform = inference_transforms()

images = [i for i in split["images"] if i["split"] in {"train", "val", "test"}]
print(f"Processing {len(images)} images...")

# Determine output dtype/model (check for old 256-d vs new 2048-d features):
# Fill with a random test image on first iteration and print shape
with torch.no_grad():
    for entry in tqdm(images):
        img_id = entry["filename"].split(".")[0]  # e.g. "COCO_train2014_000000435142"
        out_path = Path(OUTPUT_DIR) / f"{img_id}.npy"
        if out_path.exists():
            continue

        # Normalize path: map to COCO 2017
        original_filepath = entry["filepath"]  # "train2014"
        mapped_filepath = COCO_YEAR_MAP.get(original_filepath, original_filepath)
        filename = entry["filename"]  # "COCO_train2014_000000435142.jpg"
        # Strip year prefix to get clean numeric ID: "000000435142.jpg"
        numeric_filename = filename.split("_", 2)[-1]  # "000000435142.jpg"
        img_path = Path(COCO_ROOT) / mapped_filepath / numeric_filename

        if not img_path.exists():
            print(f"Missing: {img_path}")
            continue

        img = Image.open(img_path).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        feat = encoder(x).squeeze(0).cpu().to(torch.float16).numpy()
        np.save(out_path, feat)

print(f"Done. Output: {OUTPUT_DIR}")

In [ ]:
# Build vocab.json
!python scripts/build_vocab.py \
    --karpathy-split "{KARPATHY_SPLIT}" \
    --out "{OUTPUT_DIR}/vocab.json"

In [ ]:
import os
npy_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".npy")]
print(f"{len(npy_files)} feature files generated")
if npy_files:
    feat = np.load(os.path.join(OUTPUT_DIR, npy_files[0]))
    print(f"Shape: {feat.shape} dtype: {feat.dtype}")
    print(f"First 3 files: {npy_files[:3]}")

In [ ]:
!du -sh "{OUTPUT_DIR}"

## Upload

Go to the Kaggle file browser (`/kaggle/working/features/`), select all files,
click **Upload as Dataset** → name it e.g. `coco-features-2048`.

Then mount it in your training notebook and update `FEATURES_DIR`.